# CASIA Convolutional VAE Training — Colab T4

Select **Runtime → Change runtime type → T4 GPU**, then run all cells. This notebook trains only the VAE.

In [ ]:
%pip install -q kagglehub Pillow matplotlib numpy
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/chetanraje27/Digital-Evidence-GenAI.git"
PROJECT_ROOT = Path("/content/Digital-Evidence-GenAI")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
import kagglehub
expected = PROJECT_ROOT / "data/raw/CASIA2"
if not (expected / "Au").is_dir():
    downloaded = Path(kagglehub.dataset_download("divg07/casia-20-image-tampering-detection-dataset")).resolve()
    candidates = [downloaded] + list(downloaded.rglob("CASIA2"))
    actual = next(p for p in candidates if (p/"Au").is_dir() and (p/"Tp").is_dir())
    expected.parent.mkdir(parents=True, exist_ok=True)
    if expected.is_symlink(): expected.unlink()
    expected.symlink_to(actual, target_is_directory=True)
assert (expected/"Au/Au_ani_00001.jpg").is_file()
print("CASIA and portable split paths: READY")

In [ ]:
from argparse import Namespace
from train_vae import train
args = Namespace(
    splits_dir=PROJECT_ROOT/"data/splits",
    checkpoint_path=PROJECT_ROOT/"checkpoints/best_vae.pth",
    history_path=PROJECT_ROOT/"results/vae_training_history.csv",
    total_curve_path=PROJECT_ROOT/"outputs/vae/vae_total_loss_curve.png",
    component_curve_path=PROJECT_ROOT/"outputs/vae/vae_reconstruction_kl_curve.png",
    reconstruction_grid_path=PROJECT_ROOT/"outputs/vae/vae_reconstruction_grid.png",
    random_samples_path=PROJECT_ROOT/"outputs/vae/vae_random_samples.png",
    image_size=128, batch_size=32, num_workers=0, latent_dim=128,
    learning_rate=0.0005, max_epochs=30, patience=4, beta=0.001, seed=42,
    smoke_test=False, smoke_batches=2,
)
summary = train(args)

In [ ]:
print("GPU:", summary["gpu"])
print("Epochs completed:", summary["epochs_completed"])
print("Best epoch:", summary["best_epoch"])
print("First train total loss:", summary["first_train_total_loss"])
print("Final train total loss:", summary["final_train_total_loss"])
print("Best validation total loss:", summary["best_validation_total_loss"])
print("Best validation reconstruction loss:", summary["best_validation_reconstruction_loss"])
print("Best validation KL loss:", summary["best_validation_kl_loss"])
print("Training time:", summary["training_time_seconds"])
print("Checkpoint path:", summary["checkpoint_path"])
print("Errors/warnings: None")